# Data Cleaning

Parse, validate, and clean the raw NYC crash data. Compute collision severity index (CSI).

## Load Raw Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load raw data
df = pd.read_csv("../data/raw/nyc_crashes_raw.csv", low_memory=False)
print(f"Raw rows: {len(df):,}")

## Step 1: Parse Datetime

In [ ]:
# ── 1. Parse datetime ──────────────────────────────────────────────────────────
# crash_date came in as an ISO string; crash_time is HH:MM
# We combine them into a single datetime column for temporal feature engineering later

df['crash_date'] = pd.to_datetime(df['crash_date'])
df['crash_datetime'] = pd.to_datetime(
    df['crash_date'].dt.strftime('%Y-%m-%d') + ' ' + df['crash_time'],
    errors='coerce'
)

print(f"Datetime parsed. Nulls in crash_datetime: {df['crash_datetime'].isnull().sum()}")

## Step 2: Drop Rows Without Usable GPS Coordinates

In [ ]:
# ── 2. Drop rows without usable GPS coordinates ────────────────────────────────
# Without coordinates we cannot do a spatial join — these rows are useless for our pipeline
# We document exactly how many we drop and why

before = len(df)
df = df[
    df['latitude'].notnull() &
    df['longitude'].notnull() &
    (df['latitude'] != 0) &
    (df['longitude'] != 0)
]
after = len(df)
print(f"Dropped {before - after:,} rows with missing/zero coordinates ({(before-after)/before*100:.1f}%)")
print(f"Remaining: {after:,}")

## Step 3: Drop Redundant and Low-Utility Columns

In [ ]:
# ── 3. Drop redundant and low-utility columns ──────────────────────────────────
# 'location' is just "(lat, lon)" as a string — redundant with latitude/longitude
# vehicle_type and contributing_factor cols 3-5 are >90% null and won't be primary features

cols_to_drop = [
    'location',
    'vehicle_type_code_3', 'vehicle_type_code_4', 'vehicle_type_code_5',
    'contributing_factor_vehicle_3', 'contributing_factor_vehicle_4', 'contributing_factor_vehicle_5',
    'off_street_name'  # 80%+ null, less useful than on_street_name
]
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} redundant columns. Remaining columns: {df.shape[1]}")

## Step 4: Fix Numeric Columns

In [ ]:
# ── 4. Fix numeric columns ─────────────────────────────────────────────────────
# number_of_persons_injured had 1 null — fill with 0 (no injury reported = 0)
df['number_of_persons_injured'] = df['number_of_persons_injured'].fillna(0).astype(int)

## Step 5: Clean Borough Field

In [ ]:
# ── 5. Clean borough field ─────────────────────────────────────────────────────
# Standardize capitalization; we'll fill missing values later via spatial join
df['borough'] = df['borough'].str.strip().str.title()

## Step 6: Compute Collision Severity Index (CSI)

In [ ]:
# ── 6. Compute Collision Severity Index (CSI) ──────────────────────────────────
# This is the core metric of the project.
# CSI per crash = (1 × crashes) + (2 × injuries) + (5 × fatalities)
# The weights reflect that fatalities are catastrophically worse than property damage
# We store per-crash CSI here; we'll aggregate per road segment later

df['csi'] = (
    1 +                                          # every crash contributes 1
    2 * df['number_of_persons_injured'] +
    5 * df['number_of_persons_killed']
)

print(f"\n=== CSI DISTRIBUTION ===")
print(df['csi'].describe().round(2))
print(f"Crashes with CSI > 1 (had injuries/fatalities): {(df['csi'] > 1).sum():,}")

## Step 7: Save Cleaned Data

In [ ]:
# ── 7. Save cleaned file ───────────────────────────────────────────────────────
OUTPUT = Path("../data/processed/nyc_crashes_cleaned.csv")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT, index=False)
print(f"\n✓ Cleaned data saved: {OUTPUT}")
print(f"  Final shape: {df.shape}")

# Data Cleaning

This notebook performs data preprocessing, cleaning operations and data quality checks.

In [ ]:
import pandas as pd
import numpy as np

print("Data cleaning notebook initialized")

## Data Quality Checks

Perform initial data quality assessment

In [ ]:
# Check for missing values
# print("Missing values:")
# print(df.isnull().sum())

## Data Preprocessing

Clean and preprocess the data

In [ ]:
# Remove duplicates
# df_clean = df.drop_duplicates()
# print(f"Cleaned data shape: {df_clean.shape}")